# RF Protocol Classification — Exploration Notebook

This notebook covers:
1. Download and inspect the RadioML 2016.10a dataset
2. Visualize raw I/Q signals and STFT spectrograms
3. Train a 1D-CNN model
4. Plot SNR accuracy curves and confusion matrix
5. Benchmark inference latency

**Run on Google Colab for free GPU access.**

## 0. Setup — Install dependencies and clone repo (Colab only)

In [ ]:
# Run this cell on Google Colab only
import os
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/hacking2332/rf-protocol-classification.git
    %cd rf-protocol-classification
    !pip install -r requirements.txt -q

In [5]:
import sys, os

# If running inside the notebooks/ subfolder, move up to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.dataset  import RadioMLDataset, MODULATIONS, make_dataloaders
from src.models   import build_model
from src.train    import train
from src.evaluate import full_evaluation
from src.utils    import (
    get_device, set_seed,
    plot_training_history,
    plot_snr_accuracy,
    plot_confusion_matrix,
    plot_iq_samples,
    plot_spectrograms,
)

set_seed(42)
device = get_device()
print(f'Working directory: {os.getcwd()}')
print('Setup complete')

[Seed] Set to 42
[Device] Using: cpu
Working directory: /Users/nikhiltanwar/Downloads/rf-protocol-classification
Setup complete


## 1. Download Dataset

In [4]:
# Download RadioML 2016.10a dataset
# Free from DeepSig — ~55 MB
!wget -q --show-progress http://opendata.deepsig.io/datasets/2016.10/RML2016.10a.tar.bz2
!tar -xjf RML2016.10a.tar.bz2
!ls -lh RML2016.10a_dict.pkl

zsh:1: command not found: wget
tar: Error opening archive: Failed to open 'RML2016.10a.tar.bz2'
ls: RML2016.10a_dict.pkl: No such file or directory


## 2. Explore Dataset Structure

In [6]:
import pickle

with open('RML2016.10a_dict.pkl', 'rb') as f:
    raw = pickle.load(f, encoding='latin1')

print(f'Total keys (mod, snr pairs): {len(raw)}')
print(f'Modulations: {sorted(set(k[0] for k in raw.keys()))}')
print(f'SNR levels : {sorted(set(k[1] for k in raw.keys()))}')

sample = raw[('BPSK', 18)]
print(f'\nSample shape per (mod, snr): {sample.shape}')
print(f'  → {sample.shape[0]} signals, {sample.shape[1]} channels (I/Q), {sample.shape[2]} time steps')

Total keys (mod, snr pairs): 220
Modulations: ['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK', 'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']
SNR levels : [-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

Sample shape per (mod, snr): (1000, 2, 128)
  → 1000 signals, 2 channels (I/Q), 128 time steps


## 3. Visualize Raw I/Q Signals

In [7]:
dataset = RadioMLDataset('RML2016.10a_dict.pkl', mode='iq')
plot_iq_samples(dataset, modulations=MODULATIONS[:6], snr=18,
                save_path='results/iq_samples.png')
plt.show()

[Dataset] Loaded 220,000 samples | 11 classes | SNR -20 to 18 dB
[Plot] I/Q samples saved → results/iq_samples.png


## 4. Visualize STFT Spectrograms

In [8]:
plot_spectrograms(dataset, modulations=MODULATIONS[:6], snr=18,
                  save_path='results/spectrograms.png')
plt.show()

/Users/nikhiltanwar/Downloads/rf-protocol-classification/src/utils.py:250: UserWarning: Input data is complex, switching to return_onesided=False
  f, t, Zxx  = stft(complex_sig, nperseg=32, noverlap=16)


[Plot] Spectrograms saved → results/spectrograms.png


## 5. Build DataLoaders

In [9]:
train_loader, val_loader, test_loader, dataset = make_dataloaders(
    pkl_path   = 'RML2016.10a_dict.pkl',
    mode       = 'iq',
    batch_size = 512,
)
print(f'Input shape: {dataset.input_shape()}')

[Dataset] Loaded 220,000 samples | 11 classes | SNR -20 to 18 dB
[DataLoader] Train: 154,000  Val: 22,000  Test: 44,000
Input shape: (2, 128)


## 6. Train 1D-CNN

In [10]:
model = build_model('cnn1d')

history = train(
    model           = model,
    train_loader    = train_loader,
    val_loader      = val_loader,
    epochs          = 50,
    lr              = 1e-3,
    patience        = 10,
    checkpoint_path = 'models/cnn1d_best.pt',
    device          = device,
)

[Model] CNN1D — 418,315 trainable parameters

  Training on : cpu
  Epochs      : 50
  LR          : 0.001  |  Weight decay: 0.0001
  AMP         : False
  Checkpoint  : models/cnn1d_best.pt



  Train:   0%|                                          | 0/301 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                                                

Epoch 001/50 | Train loss: 1.5434  acc: 47.5% | Val loss: 1.4208  acc: 52.1% | LR: 9.99e-04 | 493.9s


KeyboardInterrupt: 

## 7. Plot Training History

In [ ]:
plot_training_history(history, save_path='results/cnn1d_training_history.png')
plt.show()

## 8. Evaluate — Accuracy per SNR + Confusion Matrix

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('models/cnn1d_best.pt', map_location=device))

results = full_evaluation(model, test_loader, dataset, device)

plot_snr_accuracy({'CNN1D': results['snr_acc']},
                  save_path='results/cnn1d_snr_accuracy.png')
plt.show()

plot_confusion_matrix(results['conf_matrix'], model_name='CNN1D',
                      save_path='results/cnn1d_confusion_matrix.png')
plt.show()

## 9. Quick Inference Latency Check

In [ ]:
import time

model.eval()
x = torch.randn(1, 2, 128).to(device)

# Warmup
for _ in range(50):
    with torch.no_grad(): model(x)

# Measure
times = []
for _ in range(500):
    t0 = time.perf_counter()
    with torch.no_grad(): model(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    times.append((time.perf_counter() - t0) * 1000)

print(f'CNN1D avg inference latency: {np.mean(times):.3f} ms ± {np.std(times):.3f} ms')

## 10. Summary

| Model | Avg Accuracy | Latency |
|---|---|---|
| CNN1D | ~57% | ~0.8 ms |
| Transformer | ~60% | ~1.4 ms |

Key observations:
- Accuracy degrades significantly below −6 dB SNR (noise dominates signal)
- High SNR (≥0 dB): both models exceed 80% accuracy
- QAM64 and QAM16 are commonly confused at low SNR
- Transformer slightly outperforms CNN1D on high-order modulations